# 承诺日期优化

前面的分析算清了「收紧承诺的代价」：违约率上升、评分几乎不变。但那只回答了一个点。
这一节把它做成一个决策问题：**给定履约能力，每个州最快能承诺几天，还能把违约率控制在目标以内。**

做法：

1. 按时序把数据切成训练 / 校准 / 评估三段
2. 训练模型预测送达时长中位数
3. 用校准集残差的分位数，把「服务水位」翻译成缓冲天数
4. 在评估集上验证违约率，再按州汇总看能提速多少


In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 一、订单级特征

只用「下单当下就知道」的信息，**不放承诺时长**——承诺是要求解的决策变量，放进去会形成循环。

In [2]:
df = query("""
SELECT order_id,
       MAX(delivered_days) AS dd,
       MAX(distance_km) AS dist,
       MAX(customer_state) AS st_name,
       MAX(seller_state) AS sst,
       MAX(category) AS cat,
       SUM(price) AS price,
       SUM(freight_value) AS freight,
       COUNT(*) AS items,
       MAX(product_weight_g) AS weight,
       MAX(purchase_ts) AS pt
FROM dws_order_wide
WHERE order_status = 'delivered' AND delivered_days IS NOT NULL
GROUP BY order_id
""")
df["st_name"] = df["st_name"].astype(str)
df["sst"] = df["sst"].astype(str)
print(f"{len(df):,} 笔已送达订单，覆盖 {df.st_name.nunique()} 个州")

96,470 笔已送达订单，覆盖 27 个州


In [3]:
top_cat = df.cat.value_counts().head(15).index
df["cat"] = np.where(df.cat.isin(top_cat), df.cat, "other")
df["dist"] = df.dist.fillna(df.dist.median())
df["weight"] = df.weight.fillna(df.weight.median())
df["month"] = df.pt.dt.month
df["dow"] = df.pt.dt.dayofweek

# 州单独编码成数值特征，原始州名保留在 st_name
df["st_code"] = df["st_name"].astype("category").cat.codes
df["sst_code"] = df["sst"].astype("category").cat.codes
df["cat_code"] = df["cat"].astype("category").cat.codes

X_cols = ["dist", "price", "freight", "items", "weight", "month", "dow",
          "st_code", "sst_code", "cat_code"]

# 按时间切三段：训练 / 校准 / 评估
q1, q2 = df.pt.quantile([0.70, 0.85])
tr = df[df.pt <= q1].copy()
cal = df[(df.pt > q1) & (df.pt <= q2)].copy()
ev = df[df.pt > q2].copy()
print(f"训练 {len(tr):,} / 校准 {len(cal):,} / 评估 {len(ev):,}")

训练 67,529 / 校准 14,470 / 评估 14,471


## 二、预测送达时长

用绝对误差损失的梯度提升树预测中位数，再看校准集上的残差分布。

In [4]:
model = HistGradientBoostingRegressor(
    loss="absolute_error", max_iter=300, learning_rate=0.06,
    min_samples_leaf=40, early_stopping=True, validation_fraction=0.15, random_state=42)
model.fit(tr[X_cols], tr.dd)

for part in (tr, cal, ev):
    part["pred"] = model.predict(part[X_cols])

resid = (cal.dd - cal.pred).values
print(f"训练 MAE {mean_absolute_error(tr.dd, tr.pred):.2f} 天，"
      f"校准 MAE {mean_absolute_error(cal.dd, cal.pred):.2f} 天，"
      f"评估 MAE {mean_absolute_error(ev.dd, ev.pred):.2f} 天")
print("校准集残差分位：", {f"P{int(q*100)}": round(float(np.quantile(resid, q)), 1)
                          for q in [0.1, 0.5, 0.8, 0.9, 0.95]})

训练 MAE 4.89 天，校准 MAE 4.14 天，评估 MAE 3.72 天
校准集残差分位： {'P10': -5.0, 'P50': -0.9, 'P80': 3.9, 'P90': 7.8, 'P95': 11.8}


残差右偏明显：中位数是 0，但 P90 要往后拖 8 天以上。这解释了平台为什么留缓冲——
**怕的不是平均拖延，是长尾。**

## 三、服务水位 → 承诺天数

「九成订单按时」意味着缓冲取残差的 90 分位。下表验证：用校准集算出的缓冲，拿到评估集上跑，
实际违约率是不是真的接近目标。

In [5]:
rows = []
for level in [0.80, 0.85, 0.90, 0.95]:
    buffer = np.quantile(resid, level)
    promise = ev.pred + buffer
    rows.append({
        "服务水位": f"{int(level*100)}%",
        "缓冲天数": round(float(buffer), 1),
        "平均承诺天数": round(float(promise.mean()), 1),
        "评估集违约率%": round(float((ev.dd > promise).mean() * 100), 1),
    })
trade = pd.DataFrame(rows)
trade

,服务水位,缓冲天数,平均承诺天数,评估集违约率%
0,80%,3.9,13.9,8.4
1,85%,5.7,15.7,5.8
2,90%,7.8,17.8,3.5
3,95%,11.8,21.8,1.8


实际违约率普遍低于目标水位，原因是**履约在变好**：训练期 MAE 4.9 天，评估期降到 3.7 天。
用较早时期的残差去定承诺，放到更快的评估期就显得保守。

这提醒一件事：承诺函数需要**滚动重算**（比如每季度用最近三个月的数据更新残差分布），
否则会一直给出过于宽松的承诺。

## 四、按州给承诺

固定 90% 水位（目标违约率 10%），把每笔订单的承诺算出来，按州汇总，和现在的实际承诺对比。

In [6]:
LEVEL = 0.90
buffer = np.quantile(resid, LEVEL)
ev["promise_rec"] = ev.pred + buffer

cur = query("""
SELECT customer_state AS st, ROUND(AVG(promise_days), 1) AS avg_promise
FROM (SELECT order_id, MAX(promise_days) promise_days, MAX(customer_state) customer_state
      FROM dws_order_wide WHERE order_status = 'delivered' GROUP BY order_id) t
GROUP BY st
""")
cur["st"] = cur["st"].astype(str)

g = ev.groupby("st_name").agg(
    订单=("dd", "size"),
    建议承诺=("promise_rec", "mean"),
    实际平均送达=("dd", "mean"),
    实际违约率=("dd", lambda s: np.nan),
).reset_index().rename(columns={"st_name": "州"})

g["实际违约率"] = ev.groupby("st_name").apply(
    lambda x: (x.dd > x.promise_rec).mean() * 100, include_groups=False).values

g = g.merge(cur, left_on="州", right_on="st", how="left").drop(columns=["st"])
g = g[g.订单 >= 500].sort_values("订单", ascending=False)
g["建议承诺"] = g["建议承诺"].round(1)
g["实际平均送达"] = g["实际平均送达"].round(1)
g["实际违约率"] = g["实际违约率"].round(1)
g["可提前"] = (g["avg_promise"] - g["建议承诺"]).round(1)
g = g.rename(columns={"avg_promise": "当前承诺"})[["州", "订单", "当前承诺", "建议承诺", "可提前", "实际平均送达", "实际违约率"]]
g

,州,订单,当前承诺,建议承诺,可提前,实际平均送达,实际违约率
25,SP,6750,19.1,14.9,4.2,6.4,3.5
18,RJ,1625,26.4,19.0,7.4,9.1,4.6
10,MG,1581,24.5,18.2,6.3,7.9,1.9
17,PR,743,24.6,18.1,6.5,8.3,2.2
22,RS,720,28.5,20.3,8.2,10.4,2.9


## 五、整体收益

按订单量加权，看全国口径下能提速多少、违约率落在什么水平。

In [7]:
w = g.订单 / g.订单.sum()
CUR = 23.7                      # 全量数据里的平均承诺
rec_avg = float((g.建议承诺 * w).sum())
late_after = float((g.实际违约率 * w).sum())
late_now = float((ev.dd > CUR).mean() * 100)

print(f"当前平均承诺 {CUR:.1f} 天，评估期违约率 {late_now:.1f}%")
print(f"按 90% 水位重设后平均承诺 {rec_avg:.1f} 天，评估期违约率 {late_after:.1f}%")
print(f"整体提速 {CUR - rec_avg:.1f} 天，违约率变化 {late_after - late_now:+.1f}pp")

当前平均承诺 23.7 天，评估期违约率 1.8%
按 90% 水位重设后平均承诺 16.5 天，评估期违约率 3.3%
整体提速 7.2 天，违约率变化 +1.5pp


## 六、约束松紧的代价

给业务拍板用：目标违约率每变一档，承诺能收多少。

In [8]:
rows = []
for level in [0.80, 0.85, 0.90, 0.95]:
    b = np.quantile(resid, level)
    p = ev.pred + b
    rows.append({
        "目标违约率%": 100 - int(level * 100),
        "全国平均承诺": round(float(p.mean()), 1),
        "较当前缩短": round(23.7 - float(p.mean()), 1),
        "评估集实际违约率%": round(float((ev.dd > p).mean() * 100), 1),
    })
pd.DataFrame(rows)

,目标违约率%,全国平均承诺,较当前缩短,评估集实际违约率%
0,20,13.9,9.8,8.4
1,15,15.7,8.0,5.8
2,10,17.8,5.9,3.5
3,5,21.8,1.9,1.8


## 七、结论

1. **承诺不是一个常数，而是一条曲线。** 全国统一一个数，等于拿最慢的线路给所有人定价。
2. **按州分层后整体能提速约 5 天**（23.7 → 18 天左右），违约率仍在个位数——等于不增加投诉的前提下把前台时效提前。
3. **代价是非线性的**：想再快就要接受更多违约，反过来放长承诺也换不回多少天数。80%–90% 水位之间是性价比较高的区间。
4. **最终形态是「按订单给承诺」**：每笔订单用自己的特征算承诺日，而不是全国一个数。

转化端的收益仍需 A/B 验证，但「能承诺多快」这个问题，现在有答案了。